In [0]:
"""
03_supplier_dimension.py

Supplier Dimension (SCD Type 2)

Source:
    material_events

Target:
    supplier_dimension

Author:
Sumanth Vempalle

Version:
2.2.0
"""

import dlt

from pyspark.sql.functions import (
    col,
    lit,
)

# ============================================================
# Supplier Source View
# ============================================================

@dlt.view(
    name="supplier_dimension_source",
    comment="Source view for Supplier Dimension."
)
def supplier_dimension_source():

    return (

        spark.readStream.table(
            "material_events"
        )

        .select(

            col("supplier"),

            # Placeholder attributes
            lit("Unknown").alias("supplier_country"),

            lit("Raw Material Supplier").alias(
                "supplier_category"
            ),

            lit("ACTIVE").alias(
                "supplier_status"
            ),

            col("event_timestamp")
                .alias("last_updated"),

        )

        .filter(
            col("supplier").isNotNull()
        )

        .dropDuplicates(
            [
                "supplier",
                "last_updated",
            ]
        )

    )


# ============================================================
# Target Streaming Table
# ============================================================

dlt.create_streaming_table(

    name="supplier_dimension",

    comment="Supplier Dimension (SCD Type 2)."

)


# ============================================================
# AUTO CDC FLOW
# ============================================================

dlt.create_auto_cdc_flow(

    target="supplier_dimension",

    source="supplier_dimension_source",

    keys=[
        "supplier",
    ],

    sequence_by="last_updated",

    stored_as_scd_type=2,

    track_history_except_column_list=[
        "last_updated",
    ],

)